In [1]:
import pandas as pd
import numpy as np
import os
import glob
import random
from datetime import timedelta

In [2]:
print("=== 階段 1：環境設定與檔案載入 ===")
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

input_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")
output_dir = input_dir # 輸出在同一個資料夾

search_pattern = os.path.join(input_dir, "reduced_final_*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    raise FileNotFoundError(f"❌ 找不到 reduced_final 檔案，請確認前一階段有成功執行！")

latest_csv = max(csv_files, key=os.path.getmtime)
date_val = os.path.basename(latest_csv).replace("reduced_final_", "").replace(".csv", "")

print(f"🤖 成功鎖定最新檔案: {os.path.basename(latest_csv)}")

=== 階段 1：環境設定與檔案載入 ===
🤖 成功鎖定最新檔案: reduced_final_0531.csv


In [3]:
df = pd.read_csv(latest_csv, low_memory=False)
print(f"📊 原始資料維度: {df.shape}")

📊 原始資料維度: (78174, 63)


In [4]:
try:
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
except ValueError:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"⏱️ 初始時間範圍: {df['timestamp'].min()} 到 {df['timestamp'].max()}\n")

⏱️ 初始時間範圍: 2026-05-30 17:32:09.790560961 到 2026-05-31 00:39:05.154409647



In [5]:
print("=== 階段 2：分析連續序列 ===")
sequences = []
sequence_value = None
sequence_start_index = None

for index, row in df.iterrows():
    if row['attack'] == sequence_value:
        continue
    else:
        if sequence_value is not None:
            sequences.append((sequence_value, list(range(sequence_start_index, index))))
        sequence_value = row['attack']
        sequence_start_index = index

if sequence_value is not None:
    sequences.append((sequence_value, list(range(sequence_start_index, len(df)))))

=== 階段 2：分析連續序列 ===


In [6]:
# 篩選並分類序列
lists_0 = [seq for val, seq in sequences if val == 0] # 正常序列
lists_1 = [seq for val, seq in sequences if val != 0] # 攻擊序列

print(f"✅ 共發現 {len(lists_0)} 段正常序列 (Label=0)")
print(f"✅ 共發現 {len(lists_1)} 段攻擊序列 (Label=1)\n")

✅ 共發現 84 段正常序列 (Label=0)
✅ 共發現 84 段攻擊序列 (Label=1)



In [7]:
# 計算各序列的持續時間
def calculate_durations(seq_lists):
    durations = []
    for seq in seq_lists:
        timestamps = df.loc[seq, 'timestamp']
        duration = timestamps.max() - timestamps.min()
        durations.append((seq, duration, len(seq)))
    return durations

normal_durations = calculate_durations(lists_0)
attack_durations = calculate_durations(lists_1)

In [8]:
print("=== 階段 3：隨機時間裁切 (消除週期性) ===")

def cut_seq_indices_atk(timestamps, sequence, lenght, new_duration):
    pivot = lenght // 2
    stop = False
    temp_duration = timestamps.loc[sequence[pivot]] - timestamps.loc[sequence[0]]
    while temp_duration < new_duration:
        stop = True
        pivot = min(pivot + 1, lenght - 1)
        temp_duration = timestamps.loc[sequence[pivot]] - timestamps.loc[sequence[0]]
    while temp_duration > new_duration and not stop:   
        pivot = max(pivot - 1, 0)
        temp_duration = timestamps.loc[sequence[pivot]] - timestamps.loc[sequence[0]]
    return sequence[0:pivot], temp_duration, len(sequence[0:pivot])

def cut_seq_indices_norm(timestamps, sequence, lenght, new_duration):
    pivot = lenght // 2
    stop = False
    temp_duration = timestamps.loc[sequence[-1]] - timestamps.loc[sequence[pivot]]
    while temp_duration < new_duration:
        stop = True
        pivot = max(pivot - 1, 0)
        temp_duration = timestamps.loc[sequence[-1]] - timestamps.loc[sequence[pivot]]
    while temp_duration > new_duration and not stop:   
        pivot = min(pivot + 1, lenght - 1)
        temp_duration = timestamps.loc[sequence[-1]] - timestamps.loc[sequence[pivot]]
    return sequence[pivot:-1], temp_duration, len(sequence[pivot:-1])

def randomize_sequence_duration(sequences_durations, percentual, cut_type):
    new_duration_sequences = []
    total_cut_seconds = 0
    for n_uple in sequences_durations:
        percent_to_cut = random.randint(1, percentual)
        dur_to_cut = (n_uple[1].total_seconds() * percent_to_cut) / 100
        new_dur = n_uple[1].total_seconds() - dur_to_cut
        timestamps = df.loc[n_uple[0], 'timestamp']

        new_sequence = n_uple[0]
        new_duration = timedelta(seconds=new_dur)
        new_lenght = n_uple[2]
        
        # 原作設定：長度 > 100 筆資料的序列才執行裁切
        if n_uple[2] > 100:
            if cut_type == 'normal':
                new_sequence, new_duration, new_lenght = cut_seq_indices_norm(timestamps, n_uple[0], n_uple[2], timedelta(seconds=new_dur))
            else:
                new_sequence, new_duration, new_lenght = cut_seq_indices_atk(timestamps, n_uple[0], n_uple[2], timedelta(seconds=new_dur))
            total_cut_seconds += dur_to_cut
            
        new_duration_sequences.append((new_sequence, new_duration, new_lenght))
    
    print(f"✂️ {cut_type} 序列總計裁切了 {total_cut_seconds:.2f} 秒的資料。")
    return new_duration_sequences

=== 階段 3：隨機時間裁切 (消除週期性) ===


In [9]:
new_attack_seqs = randomize_sequence_duration(attack_durations, 70, 'attack')
new_normal_seqs = randomize_sequence_duration(normal_durations, 30, 'normal')
print("\n")

✂️ attack 序列總計裁切了 5331.24 秒的資料。
✂️ normal 序列總計裁切了 1774.35 秒的資料。




In [10]:
print("=== 階段 4：重塑連續時間軸 ===")

merged_list_flat = []
for seq, dur, length in new_attack_seqs:
    merged_list_flat.extend(seq)
for seq, dur, length in new_normal_seqs:
    merged_list_flat.extend(seq)

# 確保索引按照原始順序排列，維持時序性
merged_list_flat.sort()

# 取出保留的資料列
df_filtered = df.loc[merged_list_flat].copy().reset_index(drop=True)
print(f"📊 裁切後資料維度: {df_filtered.shape} (移除了 {df.shape[0] - df_filtered.shape[0]} 筆冗餘資料)")

=== 階段 4：重塑連續時間軸 ===
📊 裁切後資料維度: (61529, 63) (移除了 16645 筆冗餘資料)


In [11]:
# 重新計算過濾後的連續序列區段
sequences_filtered = []
sequence_value = None
sequence_start_index = None

for index, row in df_filtered.iterrows():
    if row['attack'] == sequence_value:
        continue
    else:
        if sequence_value is not None:
            sequences_filtered.append((sequence_value, list(range(sequence_start_index, index))))
        sequence_value = row['attack']
        sequence_start_index = index
if sequence_value is not None:
    sequences_filtered.append((sequence_value, list(range(sequence_start_index, len(df_filtered)))))

In [12]:
# 重新計算過濾後的連續序列區段
sequences_filtered = []
sequence_value = None
sequence_start_index = None

for index, row in df_filtered.iterrows():
    if row['attack'] == sequence_value:
        continue
    else:
        if sequence_value is not None:
            sequences_filtered.append((sequence_value, list(range(sequence_start_index, index))))
        sequence_value = row['attack']
        sequence_start_index = index
if sequence_value is not None:
    sequences_filtered.append((sequence_value, list(range(sequence_start_index, len(df_filtered)))))

print("⏱️ 正在無縫接合時間軸 (消除間隙與重疊)...")

# 將起始時間放在迴圈外面，讓時間可以不斷「累積」下去
current_time = pd.Timestamp('2026-01-01 00:00:00')

for seq_label, seq_indices in sequences_filtered:
    if len(seq_indices) > 1:
        time_deltas = df_filtered['timestamp'].diff().iloc[seq_indices[1:]].dt.total_seconds().values
        delta = pd.Timedelta(seconds=time_deltas[0]) if not pd.isna(time_deltas[0]) else pd.Timedelta(seconds=0)
    else:
        delta = pd.Timedelta(seconds=0)

    if delta.total_seconds() == 0:
        df_filtered.loc[seq_indices, 'timestamp'] = current_time
        # 即使微小，也要稍微推進時間，避免完全重疊
        current_time += pd.Timedelta(milliseconds=1)
    else:
        new_timestamps = pd.date_range(start=current_time, periods=len(seq_indices), freq=delta)
        df_filtered.loc[seq_indices, 'timestamp'] = new_timestamps
        # 將 current_time 推進到這個區塊的最後一個時間點 + delta，作為下一個區塊的起點
        current_time = new_timestamps[-1] + delta

print(f"⏱️ 重塑後時間範圍: {df_filtered['timestamp'].min()} 到 {df_filtered['timestamp'].max()}\n")

# 將 timestamp 轉回 Unix Timestamp Float (秒)，這對機器學習模型更友善
df_filtered['timestamp'] = (df_filtered['timestamp'] - pd.Timestamp("1970-01-01")) / pd.Timedelta('1s')

⏱️ 正在無縫接合時間軸 (消除間隙與重疊)...
⏱️ 重塑後時間範圍: 2026-01-01 00:00:00 到 2026-01-02 05:08:40.914277072



In [13]:
print("=== 階段 5：輸出與存檔 ===")
final_output_path = os.path.join(output_dir, f"noperiodicity_final_{date_val}.csv")
df_filtered.to_csv(final_output_path, index=False)

print(f"✅ 成功！去週期性後的最終訓練集已存至:")
print(f"📁 {final_output_path}")

=== 階段 5：輸出與存檔 ===
✅ 成功！去週期性後的最終訓練集已存至:
📁 c:\Users\chuni\Desktop\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\4_reduced_and_noperiodicity_dataset\noperiodicity_final_0531.csv


In [14]:
import pandas as pd
import numpy as np
import os
import glob

print("🕵️ 啟動『去週期性資料』最終健康與邏輯驗證...\n")

# ==========================================
# 1. 自動定位最新檔案
# ==========================================
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

check_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")
search_pattern = os.path.join(check_dir, "noperiodicity_final_*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    print(f"❌ 找不到檔案！請確認路徑 {check_dir} 中有資料。")
else:
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"📄 正在驗證檔案: {os.path.basename(latest_csv)}")
    print("-" * 50)

    df = pd.read_csv(latest_csv, low_memory=False)

    # --- 檢查 1：維度與空值 ---
    print("📊 [檢查 1] 基本維度與純淨度")
    print(f"   - 總筆數: {df.shape[0]} 筆 (請確認比上一步的 reduced 檔案少)")
    print(f"   - 總欄位: {df.shape[1]} 個")
    
    nan_count = df.isnull().sum().sum()
    if nan_count == 0:
        print("   🟢 完美！全檔案無任何空值 (NaN)。")
    else:
        print(f"   🔴 錯誤！發現 {nan_count} 個空值。")

    string_cols = df.select_dtypes(include=['object']).columns.tolist()
    if len(string_cols) == 0:
        print("   🟢 完美！所有特徵皆為數值，模型可直接讀取。")
    else:
        print(f"   🔴 錯誤！殘留字串欄位: {string_cols}")

    # --- 檢查 2：標籤狀態 ---
    print("\n🎯 [檢查 2] 攻擊標籤 (Attack Label)")
    if 'attack' in df.columns:
        counts = df['attack'].value_counts().to_dict()
        print(f"   - 標籤分佈: {counts}")
        if set(counts.keys()).issubset({0, 1}):
            print("   🟢 完美！標籤只包含 0 (正常) 與 1 (攻擊)。")
        else:
            print(f"   🔴 錯誤！標籤異常: {counts.keys()}")
    else:
        print("   🔴 致命錯誤！找不到 attack 欄位。")

    # --- 檢查 3：時間軸健康度 ---
    print("\n⏱️ [檢查 3] 時間軸 (Timestamp) 狀態")
    if 'timestamp' in df.columns:
        is_increasing = df['timestamp'].is_monotonic_increasing
        if is_increasing:
            print("   🟢 完美！時間軸嚴格遞增，沒有發生時光倒流。")
        else:
            print("   🔴 錯誤！時間軸沒有依照順序遞增，時序模型會大亂！")
            
        is_numeric = pd.api.types.is_numeric_dtype(df['timestamp'])
        if is_numeric:
            print("   🟢 完美！時間軸是純數字 (Unix Float)，格式正確。")
        else:
            print("   🔴 錯誤！時間軸不是數字型態。")
    else:
        print("   🔴 致命錯誤！找不到 timestamp 欄位。")

    # --- 檢查 4：去週期性 (核心邏輯驗證) ---
    print("\n✂️ [檢查 4] 核心邏輯驗證：週期性是否被打破？")
    # 計算連續相同標籤的區塊
    df['block'] = (df['attack'] != df['attack'].shift(1)).cumsum()
    # 計算每個區塊的持續時間 (秒)
    seq_durations = df.groupby(['block', 'attack'])['timestamp'].apply(lambda x: x.max() - x.min()).reset_index()
    
    normal_durs = seq_durations[seq_durations['attack'] == 0]['timestamp']
    attack_durs = seq_durations[seq_durations['attack'] == 1]['timestamp']
    
    print(f"   - 正常序列 (0) 的時間長度分佈: 從 {normal_durs.min():.1f} 秒 到 {normal_durs.max():.1f} 秒")
    print(f"   - 攻擊序列 (1) 的時間長度分佈: 從 {attack_durs.min():.1f} 秒 到 {attack_durs.max():.1f} 秒")
    
    # 檢查是否有多種不同的長度
    if len(normal_durs.unique()) > 5 and len(attack_durs.unique()) > 5:
        print("   🟢 完美！正常與攻擊的時間長度變化豐富，成功打破原有的固定週期。LSTM 無法再靠『數秒數』作弊了！")
    else:
        print("   ⚠️ 警告！時間長度變化太少，可能還是有週期性特徵。")
        
    print("-" * 50)
    print("✅ 只要上方全綠燈，恭喜你！資料清洗與特徵工程大功告成，這份資料已經具備實戰等級，可以直接送進 LSTM/RNN 模型訓練了！")

🕵️ 啟動『去週期性資料』最終健康與邏輯驗證...

📄 正在驗證檔案: noperiodicity_final_0531.csv
--------------------------------------------------
📊 [檢查 1] 基本維度與純淨度
   - 總筆數: 61529 筆 (請確認比上一步的 reduced 檔案少)
   - 總欄位: 63 個
   🟢 完美！全檔案無任何空值 (NaN)。
   🟢 完美！所有特徵皆為數值，模型可直接讀取。

🎯 [檢查 2] 攻擊標籤 (Attack Label)
   - 標籤分佈: {0: 49823, 1: 11706}
   🟢 完美！標籤只包含 0 (正常) 與 1 (攻擊)。

⏱️ [檢查 3] 時間軸 (Timestamp) 狀態
   🟢 完美！時間軸嚴格遞增，沒有發生時光倒流。
   🟢 完美！時間軸是純數字 (Unix Float)，格式正確。

✂️ [檢查 4] 核心邏輯驗證：週期性是否被打破？
   - 正常序列 (0) 的時間長度分佈: 從 0.0 秒 到 56557.8 秒
   - 攻擊序列 (1) 的時間長度分佈: 從 0.0 秒 到 1.2 秒
   🟢 完美！正常與攻擊的時間長度變化豐富，成功打破原有的固定週期。LSTM 無法再靠『數秒數』作弊了！
--------------------------------------------------
✅ 只要上方全綠燈，恭喜你！資料清洗與特徵工程大功告成，這份資料已經具備實戰等級，可以直接送進 LSTM/RNN 模型訓練了！
